# Bulk YouTube Uploader
Run this notebook in Google Colab to recursively upload videos from a specific folder on your Google Drive.

In [ ]:
# @title 1. Install Dependencies
!pip install --upgrade google-api-python-client google-auth google-auth-oauthlib google-auth-httplib2


In [ ]:
# @title 2. Bulk Upload Execution
import http.client as httplib
import httplib2
import os
import random
import time
import mimetypes

from google.colab import drive

from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload

# Provide your OAuth 2.0 Credentials (Client ID and Client Secret) below.
# You can generate these in the Google Cloud Console:
# APIs & Services > Credentials > Create Credentials > OAuth client ID (Desktop app)
client_id = "" # @param {type:"string"}
client_secret = "" # @param {type:"string"}
target_folder_name = "Allen Videos" # @param {type:"string"}
privacy_status = "private" # @param ["public", "private", "unlisted"]
video_category = "22" # @param {type:"string"}
default_tags = "Education" # @param {type:"string"}

httplib2.RETRIES = 1
MAX_RETRIES = 10

RETRIABLE_STATUS_CODES = [500, 502, 503, 504]
YOUTUBE_UPLOAD_SCOPE = ["https://www.googleapis.com/auth/youtube.upload"]
YOUTUBE_API_SERVICE_NAME = "youtube"
YOUTUBE_API_VERSION = "v3"

def get_authenticated_service(client_id, client_secret):
    client_config = {
        "installed": {
            "client_id": client_id,
            "project_id": "youtube-bulk-uploader",
            "auth_uri": "https://accounts.google.com/o/oauth2/auth",
            "token_uri": "https://oauth2.googleapis.com/token",
            "auth_provider_x509_cert_url": "https://www.googleapis.com/oauth2/v1/certs",
            "client_secret": client_secret,
            "redirect_uris": ["http://localhost:8080/"]
        }
    }
    
    flow = InstalledAppFlow.from_client_config(client_config, YOUTUBE_UPLOAD_SCOPE)
    flow.redirect_uri = 'http://localhost:8080/'
    
    auth_url, _ = flow.authorization_url(prompt='consent')
    
    print('Please go to this URL and authorize the application:')
    print(auth_url)
    print('\nAfter authorizing, Google will redirect you to a "localhost" address that will fail to load.')
    print('Copy the ENTIRE URL from your browser address bar and paste it below.')
    response = input('Enter the full redirect URL: ')
    
    flow.fetch_token(authorization_response=response)
    return build(YOUTUBE_API_SERVICE_NAME, YOUTUBE_API_VERSION, credentials=flow.credentials)

def initialize_upload(youtube, file_path, title, description, tags, category, privacy):
    body=dict(
        snippet=dict(
            title=title,
            description=description,
            tags=tags,
            categoryId=category
        ),
        status=dict(
            privacyStatus=privacy
        )
    )

    insert_request = youtube.videos().insert(
        part=",".join(body.keys()),
        body=body,
        media_body=MediaFileUpload(file_path, chunksize=-1, resumable=True)
    )

    return resumable_upload(insert_request)

def resumable_upload(insert_request):
    response = None
    error = None
    retry = 0
    while response is None:
        try:
            print("Uploading file...")
            status, response = insert_request.next_chunk()
            if response is not None:
                if 'id' in response:
                    print("Video id '%s' was successfully uploaded." % response['id'])
                    return True
                else:
                    print("The upload failed with an unexpected response: %s" % response)
                    return False
        except HttpError as e:
            if e.resp.status in RETRIABLE_STATUS_CODES:
                error = f"A retriable HTTP error {e.resp.status} occurred:\n{e.content}"
            else:
                raise
        except Exception as e:
            error = f"A retriable error occurred: {e}"

        if error is not None:
            print(error)
            retry += 1
            if retry > MAX_RETRIES:
                print("No longer attempting to retry.")
                return False

            max_sleep = 2 ** retry
            sleep_seconds = random.random() * max_sleep
            print(f"Sleeping {sleep_seconds} seconds and then retrying...")
            time.sleep(sleep_seconds)
    return False

def find_target_folder(base_path, target_name):
    normalized_target = target_name.replace(" ", "").lower()
    
    print(f"Searching for folder '{target_name}' using os.walk (this may take a while depending on drive size)...")
    for root, dirs, files in os.walk(base_path):
        for d in dirs:
            if d.replace(" ", "").lower() == normalized_target:
                return os.path.join(root, d)
    return None

def is_video_file(file_path):
    mime_type, _ = mimetypes.guess_type(file_path)
    if mime_type and mime_type.startswith('video'):
        return True
    
    video_exts = ['.mp4', '.mkv', '.avi', '.mov', '.wmv', '.flv', '.webm', '.m4v']
    _, ext = os.path.splitext(file_path)
    return ext.lower() in video_exts

if not client_id or not client_secret:
    print("Error: You must provide both a client_id and a client_secret in the form fields.")
else:
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

    youtube = get_authenticated_service(client_id, client_secret)
    base_drive_path = "/content/drive/MyDrive"
    
    target_path = find_target_folder(base_drive_path, target_folder_name)

    if not target_path:
        print(f"Error: Could not find any folder matching '{target_folder_name}'.")
    else:
        print(f"Found folder at: {target_path}")
        videos_uploaded = 0
        tags_list = [tag.strip() for tag in default_tags.split(",") if tag.strip()]
        
        for root, dirs, files in os.walk(target_path):
            for file in sorted(files):
                file_path = os.path.join(root, file)
                
                if is_video_file(file_path):
                    filename_no_ext = os.path.splitext(file)[0]
                    title = filename_no_ext[:100]
                    description = f"Video uploaded from {target_folder_name}\nOriginal filename: {file}"
                    
                    print(f"\nPreparing to upload: {file_path}")
                    success = initialize_upload(
                        youtube,
                        file_path,
                        title,
                        description,
                        tags_list,
                        video_category,
                        privacy_status
                    )
                    
                    if success:
                        videos_uploaded += 1
                        
        print(f"\nBulk upload completed! Total videos uploaded: {videos_uploaded}")
